# Base R — Technical Reference

R companion to the NumPy reference. Where NumPy has `ndarray`, base R has
**vectors, matrices, and arrays** — same core ideas (shape, dtype, vectorized
ops, broadcasting), different defaults (1-indexed, copy-on-modify instead of
views, recycling instead of broadcasting).

## Quick Index

| Pattern | When to use | Section |
| :--- | :--- | :--- |
| Vector / matrix creation | Build vectors, matrices, arrays from scratch or data | §1 |
| Properties & types | Inspect shape, type, memory | §2 |
| Indexing & slicing | Subset vectors / matrices / data frames | §3 |
| Reshaping & combining | Change dimensions, combine objects | §4 |
| Math & aggregations | Element-wise ops, reductions | §5 |
| Recycling (R's "broadcasting") | Operations on objects of different lengths | §6 |
| Linear algebra | Matrix ops, decompositions, solvers | §7 |
| Random & simulation | Sampling, seeding, distributions | §8 |
| Performance | Copy-on-modify, vectorization | §9 |

---
## When to Use

| Signal | R function to reach for |
| :--- | :--- |
| Create a vector of zeros / ones | `numeric(n)`, `rep(1, n)`, `rep(7, n)` |
| Evenly spaced values | `seq(from, to, by=)` (step-based) or `seq(from, to, length.out=)` (count-based) |
| Filter elements by condition | Logical indexing `x[x > 0]` |
| Apply condition element-wise | `ifelse(cond, x, y)` |
| Aggregate across a margin | `rowSums()`, `colSums()`, `apply(m, 1, fn)` / `apply(m, 2, fn)` |
| Reshape without copying conceptually | `dim(a) <- c(r, c)` or `t()` |
| Stack arrays side by side | `cbind()` |
| Stack arrays top to bottom | `rbind()` |
| Matrix multiplication | `A %*% B` |
| Solve linear system Ax = b | `solve(A, b)` |
| Generate reproducible random data | `set.seed()` then `r*()` family |
| Check if a modification copies | R always copies on modify — there is no in-place view to check |

---
## §1 — Vector & Matrix Creation

Every R workflow starts here. Know which constructor to reach for and what type it produces by default.

In [ ]:
# From R literals
print(c(1, 2, 3))                          # numeric vector, type inferred (double)
print(matrix(c(1, 2, 3, 4), nrow = 2))     # 2x2 matrix, filled COLUMN-major by default
print(c(1L, 2L, 3L))                       # integer vector (force with L suffix)

# Filled vectors / matrices
print(numeric(12))                          # vector of 12 zeros (double)
print(rep(1, 12))                           # vector of 12 ones
print(matrix(0, nrow = 3, ncol = 4))        # 3x4 matrix of 0.0
print(diag(4))                              # 4x4 identity matrix

# Range vectors
print(seq(0, 10, by = 2))                   # 0 2 4 6 8 10
print(seq(0, 1, length.out = 5))            # 0 0.25 0.5 0.75 1.0

# From an existing vector/matrix
arr <- matrix(1:12, nrow = 3)
print(matrix(0, nrow = nrow(arr), ncol = ncol(arr)))  # same shape as arr, filled with 0


**`seq(..., by=)` vs `seq(..., length.out=)`:**

| | Control | Endpoint included | Use when |
| :--- | :--- | :--- | :--- |
| `seq(from, to, by = step)` | Step size | Only if reached exactly | You know the step |
| `seq(from, to, length.out = n)` | Count | Always | You know how many points |

**Common mistakes:**
- `matrix(1:6, 3, 4)` silently recycles/truncates if the element count doesn't divide evenly into the shape — R warns but still runs
- `seq()` with float steps can accumulate floating point error, same as `np.arange` — use `length.out` for evenly spaced floats
- `matrix(nrow=, ncol=)` with no data fills with `NA`, not arbitrary memory garbage like `np.empty` -- this is actually safer than NumPy's equivalent
- R's default `matrix()` fill order is **column-major**; pass `byrow = TRUE` to fill row-major like NumPy's default

---
## §2 — Properties & Types

Always check shape and type before operating. Silent type coercion and shape mismatches are the most common source of bugs, exactly as in NumPy.

In [ ]:
a <- matrix(1:6, nrow = 2, ncol = 3, byrow = TRUE)

cat("dim(a)          :", dim(a), "\n")
cat("length(dim(a))  :", length(dim(a)), "\n")
cat("length(a)       :", length(a), "\n")
cat("nrow(a)         :", nrow(a), "\n")
cat("typeof(a)       :", typeof(a), "\n")
cat("class(a)        :", class(a), "\n")
cat("object.size(a)  :"); print(object.size(a))
storage.mode(a) <- "double"
cat("after cast to double, typeof:", typeof(a), "\n")


**Common mistakes:**
- Mixing integer and double vectors -- R upcasts silently: `c(1L, 2L) + c(0.5)` returns `double`, same idea as NumPy's int->float upcast
- Using `nrow(a)` thinking it is total count -- it is rows only; use `length(a)` for the NumPy `.size` equivalent
- Forgetting `storage.mode(a) <- "double"` modifies in place via the replacement function -- unlike `a.astype()` in NumPy, R's "casting" functions like `as.integer()` do NOT preserve dimensions automatically (see SS4)

---
## §3 — Indexing & Slicing

R indexing is **1-based**, and unlike NumPy, indexing a vector or matrix in R **always copies** (R has copy-on-modify semantics, not views). See SS9 for what this means for performance.

In [ ]:
a <- matrix(1:9, nrow = 3, byrow = TRUE)
cat("matrix a:\n"); print(a)

cat("\nFirst row a[1,]         :", a[1, ], "\n")
cat("Last row  a[nrow(a),]   :", a[nrow(a), ], "\n")
cat("a[2,3]                  :", a[2, 3], "\n")
cat("a[,2] (col 2)           :", a[, 2], "\n")
cat("a > 5 (logical filter)  :", a[a > 5], "\n")

cat("\nSubmatrix a[1:2, 2:3]:\n"); print(a[1:2, 2:3])
cat("which(a > 5):\n"); print(which(a > 5, arr.ind = TRUE))
cat("ifelse(a > 5, a, 0):\n"); print(ifelse(a > 5, a, 0))


**Views vs copies -- the single biggest conceptual difference from NumPy:**

| Operation | NumPy | R |
| :--- | :--- | :--- |
| Basic slicing | View (shares memory) | **Copy** (always) |
| Logical indexing | Copy | Copy |
| Index-vector indexing | Copy | Copy |

R has **no view concept** for ordinary indexing -- every `a[...]` read produces an independent copy, and modifying that copy never touches `a`. This removes an entire category of NumPy bugs (accidentally mutating an array via a slice) but means you cannot rely on slicing being free; see SS9 for the actual cost model (copy-on-*modify*, not copy-on-*read*).

**Common mistakes:**
- Assuming 0-based indexing out of NumPy habit -- `a[0, ]` in R either errors or returns an empty result depending on context; R starts at 1
- Forgetting that `a[2:3, ]` includes both endpoints -- R ranges are inclusive on both ends, unlike Python slices
- `which(cond)` alone returns flat positions, not a value-substitution result -- use `ifelse(cond, x, y)` for value substitution, same caveat as `np.where(cond)` vs `np.where(cond, x, y)`

---
## §4 — Reshaping & Combining

Reshaping in R is done via the `dim<-` replacement function (in place, no copy concept to reason about since R always copies on write). Use `cbind`/`rbind` to combine objects along existing or new axes.

In [ ]:
a <- 0:11
cat("a =", a, "\n\n")

cat("matrix(a, 3, 4) -- column-major:\n");  print(matrix(a, nrow = 3, ncol = 4))
cat("t(m) -- transpose:\n"); m <- matrix(a, nrow = 3, ncol = 4); print(t(m))
cat("as.vector(m) -- flatten:\n"); print(as.vector(m))

cat("\nrbind(x, y):\n"); x <- c(1,2,3); y <- c(4,5,6); print(rbind(x, y))
cat("cbind(x, y):\n"); print(cbind(x, y))
cat("c(x, y):", c(x, y), "\n")

cat("\nrep(x, each=2) :", rep(x, each = 2), "\n")
cat("rep(x, times=2):", rep(x, times = 2), "\n")


**`rbind`/`cbind` vs `c()` vs `abind::abind`:**

| | Creates new axis? | Use when |
| :--- | :--- | :--- |
| `rbind` / `cbind` | Yes (for vectors), No (for matrices of compatible shape) | Combining 1D vectors into a 2D matrix, or stacking matrices |
| `c()` | No | Combining along the existing (only) axis of plain vectors |
| `abind::abind(..., along=)` | Depends on `along` | nD generalization, R's `np.stack` / `np.concatenate` equivalent |

**Common mistakes:**
- `matrix(a, nrow=, ncol=)` recycles silently with a warning if the element count doesn't match -- always check `length(a)` before reshaping, same discipline as checking `a.size` in NumPy
- R fills matrices **column-major** by default; NumPy fills **row-major** by default -- always double check with `byrow = TRUE` when porting layouts from Python
- `as.vector()`/`c()` always copy, matching `np.flatten()`'s always-copy behavior -- there is no `ravel()`-style "view if possible" option in R, because nothing is ever a view

**`rep()` variants for the NumPy `repeat`/`tile` distinction:**

| | What it does | NumPy equivalent |
| :--- | :--- | :--- |
| `rep(x, each = n)` | repeat **each element** n times | `np.repeat(x, n)` |
| `rep(x, times = n)` | repeat the **whole vector** n times | `np.tile(x, n)` |

In [ ]:
x <- c(1, 2, 3)
cat("rep(x, each=2) :", rep(x, each = 2), "\n")
cat("rep(x, times=2):", rep(x, times = 2), "\n")


---
## §5 — Math & Aggregations

All operations are element-wise by default, exactly as in NumPy. Use `apply()`, `rowSums()`/`colSums()`, or `rowMeans()`/`colMeans()` to reduce along a specific margin.

In [ ]:
a <- matrix(1:6, nrow = 2, byrow = TRUE)
cat("a:\n"); print(a)
cat("a + 10:\n");    print(a + 10)
cat("a * 2:\n");     print(a * 2)
cat("sqrt(a):\n");   print(round(sqrt(a), 3))
cat("colSums(a):", colSums(a), "\n")
cat("rowSums(a):", rowSums(a), "\n")
cat("sum(a)    :", sum(a), "\n")
cat("mean(a)   :", mean(a), "\n")
cat("sd(a)     :", round(sd(a), 4), "  (note: R uses n-1 divisor)\n")

A <- matrix(c(1,2,3,4), nrow=2, byrow=TRUE)
B <- matrix(c(5,6,7,8), nrow=2, byrow=TRUE)
cat("\nA %*% B (matrix multiply):\n"); print(A %*% B)
cat("A * B (element-wise):\n");        print(A * B)


**Margin mental model (R's `MARGIN` argument, same idea as NumPy's `axis`):**
- `MARGIN = 1` (rows) -> apply/reduce **across columns for each row** -- conceptually NumPy's `axis=1`
- `MARGIN = 2` (columns) -> apply/reduce **down rows for each column** -- conceptually NumPy's `axis=0`
- This is the **opposite numbering convention** from NumPy's axis -- the single most common source of bugs when porting code

**Common mistakes:**
- `A * B` is element-wise, not matrix multiplication -- use `A %*% B` for matrix multiply, same trap as NumPy's `A * B` vs `A @ B`
- R's `sd()`/`var()` use the **sample** formula (divide by n-1); NumPy's `.std()`/`.var()` default to the **population** formula (divide by n) -- pass `ddof=1` in NumPy or compute manually in R to match
- R's `MARGIN=1`/`MARGIN=2` in `apply()` is the reverse of NumPy's `axis=0`/`axis=1` -- always sanity-check against a known small example when porting

**`%*%` vs `crossprod`/`tcrossprod` vs `*`:**

| | 1-D inputs | 2-D inputs | Use when |
| :--- | :--- | :--- | :--- |
| `A %*% B` | dot product (1x1 matrix) | matrix multiply | General matrix multiplication |
| `crossprod(A, B)` | -- | `t(A) %*% B`, faster | You need `A'B` -- common in regression (`X'X`) |
| `tcrossprod(A, B)` | -- | `A %*% t(B)`, faster | You need `AB'` |

**`identical()` vs `all.equal()` vs `==`:**

| | Returns | Float-safe? | Use when |
| :--- | :--- | :--- | :--- |
| `a == b` | elementwise logical vector | No | You want a per-element mask |
| `identical(a, b)` | single logical, **exact** | No | Exact equality including type/attributes |
| `isTRUE(all.equal(a, b))` | single logical, **tolerance** | Yes | Comparing floats (always use this) |

In [ ]:
# float comparison trap -- same IEEE-754 issue as NumPy
a <- c(0.1 + 0.2, 1.0)
b <- c(0.3, 1.0)
cat("a == b              :", a == b, "\n")
cat("identical(a, b)     :", identical(a, b), "\n")
cat("isTRUE(all.equal)   :", isTRUE(all.equal(a, b)), "\n")

---
## §6 — Recycling (R's "Broadcasting")

R does not call this "broadcasting", but the mechanism -- stretching a shorter object to match a longer one without copying data -- serves the same purpose. The rules are looser and less explicit than NumPy's, which is exactly why this section exists: recycling will *silently* do something unintended where NumPy would raise a `ValueError`.

In [ ]:
a <- matrix(1:6, nrow = 2, byrow = TRUE)
cat("a:\n"); print(a)
cat("\na + 10 (scalar recycles):\n"); print(a + 10)

row <- c(10, 20, 30)
cat("\na + row (R recycles column-major -- adds row DOWN cols, not across rows):\n")
print(a + row)
cat("Correct row-broadcast via t(t(a) + row):\n")
print(t(t(a) + row))

col <- c(10, 20)
cat("\na + col (length matches nrow -- works as expected):\n")
print(a + col)

x <- c(1, 2, 3); y <- c(10, 20)
cat("\nouter(y, x):\n"); print(outer(y, x))

sweep_result <- sweep(a, MARGIN = 1, STATS = rowMeans(a), FUN = "-")
cat("\nsweep(): subtract each row's mean:\n"); print(sweep_result)


**Recycling vs NumPy broadcasting -- the critical difference:**

| | NumPy | R |
| :--- | :--- | :--- |
| Mismatched, non-multiple lengths | Raises `ValueError` | Recycles anyway, only a **warning** |
| Matrix + short vector direction | Explicit per-axis rules (`axis=` aware) | Always recycles **column-major**, regardless of intent |
| Safe way to add a row vector to every row | `a + row` just works | Must use `sweep()` or `t(t(a) + row)` |
| Safe way to add a column vector to every column | `a + col[:, None]` | `a + col` (matches R's natural column-major recycling) |

**Common mistakes:**
- Assuming `matrix + row_vector` adds the vector to each row like NumPy -- R recycles column-major, so it actually adds the vector down each column first, only "looking right" by coincidence for square-ish cases. **Always use `sweep()` when intent matters.**
- Trusting that R will error on a shape mismatch -- it almost never does; mismatched-but-not-multiple lengths silently recycle with only a warning that is easy to miss in long output
- Forgetting that `outer(x, y)` (note argument order) builds an outer product explicitly -- prefer it over manual `newaxis`-style tricks for clarity

---
## §7 — Linear Algebra

All linear algebra is available in base R (no separate `linalg` submodule needed). Critical for quantitative analyst roles and statistics. Always check that your matrix is square / full-rank before inverting.

In [ ]:
A <- matrix(c(2, 1, 5, 3), nrow = 2, byrow = TRUE)
b <- c(4, 7)
S <- A %*% t(A)   # symmetric, positive-definite

cat("A:\n"); print(A)
cat("det(A)   :", det(A), "\n")
cat("rank(A)  :", qr(A)$rank, "\n")
cat("trace(A) :", sum(diag(A)), "\n")

cat("\nsolve(A) -- inverse:\n"); print(round(solve(A), 4))
cat("solve(A, b) -- solve Ax=b:", round(solve(A, b), 4), "\n")

eig <- eigen(A)
cat("\neigen(A) values :", round(eig$values, 4), "\n")
sv <- svd(A)
cat("svd(A) singular values:", round(sv$d, 4), "\n")

cat("\nchol(S) -- Cholesky (upper triangular):\n"); print(round(chol(S), 4))


**`solve(A, b)` vs `solve(A) %*% b`:**

| | Speed | Numerical stability | Use when |
| :--- | :--- | :--- | :--- |
| `solve(A, b)` | Faster | Better | Solving Ax = b |
| `solve(A) %*% b` | Slower | Worse | Only when you need A^-1 explicitly |

**`eigen()` general vs `symmetric = TRUE`:**

| | Valid input | Output | Use when |
| :--- | :--- | :--- | :--- |
| `eigen(A)` (default) | Any square matrix | Possibly complex, R auto-detects symmetry by checking the matrix | General matrix |
| `eigen(S, symmetric = TRUE)` | You assert symmetric/Hermitian | Real, faster algorithm | You already know it's symmetric -- skip R's auto-check for speed |

Unlike NumPy (separate `eig` vs `eigh` functions, where `eigh` silently returns a wrong answer on non-symmetric input), R's `eigen()` **auto-detects** symmetry by default and dispatches accordingly -- the `symmetric=` argument is purely a speed optimization, not a correctness requirement. This is a meaningfully safer default than NumPy's.

**Common mistakes:**
- Calling `solve(A)` on a singular matrix -- raises an error (`Lapack routine dgesv: system is exactly singular`); check `det(A) != 0` first, same discipline as NumPy
- R's `chol()` returns an **upper**-triangular factor (`t(L) %*% L = S`) by default, the transpose convention of NumPy's `np.linalg.cholesky` (which returns lower-triangular, `L @ L.T = S`) -- transpose if you need the NumPy convention
- Forgetting that `svd()` returns `$v` (not transposed), while NumPy's `svd` returns `Vt` (transposed) -- `A ≈ sv$u %*% diag(sv$d) %*% t(sv$v)` in R vs `A ≈ U @ diag(S) @ Vt` in NumPy

**Verified -- why `eigen()`'s auto-detection matters:**

In [ ]:
A <- matrix(c(2, 1, 5, 3), nrow = 2, byrow = TRUE)   # non-symmetric
S <- A %*% t(A)                                       # symmetric, positive-definite

cat("S symmetric?       :", isSymmetric(S), "\n")
cat("eigen(S)$values     :", sort(eigen(S)$values), "\n")              # correct
cat("eigen(S, sym=T)$val :", eigen(S, symmetric = TRUE)$values, "\n")  # correct (R's eigen sorts descending by default)
cat("eigen(A)$values      :", sort(eigen(A)$values), "\n")             # correct for general A -- R auto-detects A is NOT symmetric
# Unlike NumPy's eigh(), there is no "wrong answer" failure mode here -- eigen() checks symmetry itself unless told otherwise

---
## §8 — Random & Simulation

Use `set.seed()` for reproducibility. Unlike NumPy's modern `default_rng()` generator-object API, R's random functions are all driven by **global RNG state** -- there is no first-class "local generator" equivalent in base R (the `rngtools`/`dqrng` packages add one if you need it).

In [ ]:
set.seed(42)
cat("runif(5):", round(runif(5), 4), "\n")
cat("rnorm(5):", round(rnorm(5), 4), "\n")
cat("rbinom(5, size=10, prob=0.5):", rbinom(5, 10, 0.5), "\n")
cat("sample(1:4, 5, replace=TRUE):", sample(1:4, 5, replace=TRUE), "\n")

cat("\n3x4 matrix of uniform [0,1):\n")
print(round(matrix(runif(12), nrow=3, ncol=4), 3))

# Monte Carlo pi estimate
set.seed(0)
n <- 1e6
x <- runif(n); y <- runif(n)
inside <- (x^2 + y^2) < 1
cat(sprintf("\nMonte Carlo pi estimate (n=1M): %.5f\n", 4 * mean(inside)))


**R's global-state RNG vs NumPy's `default_rng()` generator objects:**

| | `np.random.seed()` (legacy) | `np.random.default_rng()` (modern) | R's `set.seed()` |
| :--- | :--- | :--- | :--- |
| Thread-safe | No | Yes | No (same limitation as NumPy's legacy API) |
| Reproducible | Yes (global state) | Yes (local state) | Yes (global state) |
| Local/isolated generator object | No | Yes | Not in base R |

**Common mistakes:**
- Expecting an R equivalent of `rng.shuffle()` that mutates in place -- R has none; `sample(x)` always returns a new shuffled vector, consistent with R's copy-on-modify model throughout
- `sample(0:9, ...)` -- R's `sample()` range is inclusive on both ends by default (`0:9` is 0 through 9), so match the NumPy convention deliberately when porting `rng.integers(0, 10)` (which excludes 10)
- Running parallel R processes (e.g. via `parallel::mclapply`) without `set.seed()`'s `RNGkind("L'Ecuyer-CMRG")` stream-safe mode -- same race-condition class of bug as NumPy's legacy global-state API in threaded code

---
## §9 — Performance

R's speed, like NumPy's, comes from vectorized operations implemented in C under the hood. The moment you introduce an explicit `for` loop over elements, you lose that advantage -- but R's memory model differs from NumPy's in one fundamental way: **everything copies on modify**, there are no views.

In [ ]:
# Copy-on-modify demo
a <- matrix(1:12, nrow = 3, ncol = 4)
b <- a[1:2, ]
b[1, 1] <- 99
cat("a[1,1] after modifying b:", a[1, 1], "  (unchanged -- b is a copy)\n")
cat("b[1,1]                  :", b[1, 1], "\n")

# Vectorization speed principle (demonstration, not timed)
a_vec <- 1:100000
result_vectorized <- a_vec ^ 2
cat("Vectorized: a^2 for 100k elements -- length of result:", length(result_vectorized), "\n")

# Memory layout: R is column-major
a_mat <- matrix(1, nrow = 1000, ncol = 1000)
cat("colSums shape:", length(colSums(a_mat)), "(one value per column)\n")
cat("rowSums shape:", length(rowSums(a_mat)), "(one value per row)\n")

# Batch dot product via rowSums
X <- matrix(rnorm(100), nrow = 10)
Y <- matrix(rnorm(100), nrow = 10)
dots <- rowSums(X * Y)
cat("Row-wise dot products (10 values):", round(dots, 3), "\n")


**Common mistakes:**
- Calling `.copy()`-equivalent operations defensively, the way you might in NumPy -- unnecessary in R, since every assignment is already copy-on-modify by default; there is nothing to "protect" against
- Using `sapply()`/`for` loops inside hot paths instead of vectorized operations -- same speed penalty class as Python `math` calls inside a loop instead of NumPy ufuncs
- Building large matrices and assuming float32-equivalent memory savings are available -- base R doubles are always 8 bytes; reach for the `float` package if true float32 storage matters
- Growing a vector inside a loop with `c(result, new_value)` -- this **reallocates and copies the entire vector every iteration**, an R-specific trap with no direct NumPy analog (NumPy users coming from `np.append()` in a loop have the closest intuition for why this is slow); pre-allocate with `numeric(n)` instead, as shown above

---
# Decision Guide

```
Creating a vector or matrix?
+-- From existing data                        -> c(), matrix()
+-- All zeros / ones / constant                -> numeric(n), rep(1, n), rep(k, n)
+-- Range with fixed step                      -> seq(from, to, by = step)
+-- N evenly spaced points                     -> seq(from, to, length.out = n)
+-- Identity matrix                            -> diag(n)

Indexing / filtering?
+-- Slice by position                          -> a[start:stop, ] -- ALWAYS a copy (no views in R)
+-- Filter by condition                        -> a[a > threshold] -- copy
+-- Select by index vector                     -> a[c(1, 3, 5), ] -- copy
+-- Conditional value substitution              -> ifelse(cond, x, y)
+-- Need to modify without affecting original  -> nothing extra needed -- it already won't

Changing shape?
+-- Reshape to known dimensions                -> matrix(a, nrow = r, ncol = c)
+-- Transpose                                  -> t(a)
+-- Flatten to 1D                              -> as.vector(a) / c(a) -- always a copy
+-- Add a dimension                            -> array(a, dim = c(1, n)) or array(a, dim = c(n, 1))

Combining objects?
+-- Stack as new rows                          -> rbind(a, b)
+-- Stack as new columns                       -> cbind(a, b)
+-- Along a new axis (nD)                      -> abind::abind(a, b, along = n)
+-- Along an existing axis (1D)                -> c(a, b)

Aggregating?
+-- Entire object                              -> sum(a), mean(a), etc.
+-- One value per column                       -> colSums(a) / colMeans(a)  (NumPy axis=0)
+-- One value per row                          -> rowSums(a) / rowMeans(a)  (NumPy axis=1)
+-- Custom reduction along a margin            -> apply(a, MARGIN, fn)  -- MARGIN is REVERSED vs NumPy's axis!

Linear algebra?
+-- Matrix multiplication                      -> A %*% B
+-- Solve Ax = b                               -> solve(A, b)
+-- Eigenvalues (any matrix)                   -> eigen(A)  -- auto-detects symmetry
+-- Eigenvalues (symmetric, faster)            -> eigen(A, symmetric = TRUE)
+-- SVD                                        -> svd(A)
+-- Inverse (use sparingly)                    -> solve(A)

Performance?
+-- Slow explicit loop over a vector           -> replace with a vectorized op
+-- Growing a vector inside a loop             -> pre-allocate with numeric(n) first
+-- Need genuinely independent data            -> nothing extra -- R already copies on modify
+-- Complex multi-axis contraction              -> rowSums(X * Y) patterns, or the `einsum` package
```